# LeetCode 435 — Non-overlapping Intervals bằng Python

Link đề: https://leetcode.com/problems/non-overlapping-intervals/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Dynamic Programming, Sorting

Skeleton LeetCode:

```python
class Solution:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết lại theo hiểu biết về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code (vét cạn mọi tập con), nhưng nên đối chiếu lại đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho nhiều **đoạn** `[start, end]`, cần **xoá ít đoạn nhất** để các đoạn còn lại **không chồng lấn** (hai đoạn chỉ chạm nhau tại một điểm như `[1,2]` và `[2,3]` vẫn được coi là **không** chồng lấn).
2. Nắm các công cụ Python: `list.sort(key=...)`, `lambda`, giải nén `for l, r in intervals`, `float('-inf')`, `bisect`, và duyệt tập con bằng **bitmask** (`1 << n`).
3. Đi từ **vét cạn mọi tập con** → **DP kiểu LIS `O(n²)`** → **tham lam chọn đoạn kết thúc sớm nhất** → **tham lam theo điểm bắt đầu, gặp chồng lấn thì bỏ đoạn kết thúc muộn** → **DP + tìm kiếm nhị phân `O(n log n)`**.
4. Hiểu mấu chốt: "xoá ít nhất" tương đương "**giữ nhiều nhất**" các đoạn đôi một rời nhau, và đoạn **kết thúc càng sớm** càng để lại nhiều chỗ cho các đoạn sau (**exchange argument**).
5. Tránh các lỗi kinh điển: sắp theo **điểm bắt đầu** rồi giữ đoạn đầu tiên, dùng `>` thay `>=` (coi đoạn chạm nhau là chồng lấn), quên rằng `sort` **thay đổi mảng đầu vào**.

## 2. Đề bài & ví dụ

Cho mảng `intervals` trong đó `intervals[i] = [start_i, end_i]`. Hãy trả về **số đoạn tối thiểu cần xoá** để các đoạn còn lại **không chồng lấn**.

**Lưu ý:** hai đoạn chỉ **chạm nhau tại một điểm** là **không chồng lấn**. Ví dụ `[1, 2]` và `[2, 3]` là hợp lệ.

**Ví dụ 1**

- Input: `intervals = [[1,2],[2,3],[3,4],[1,3]]`
- Output: `1`
- Giải thích: xoá `[1,3]` thì còn `[1,2],[2,3],[3,4]` — các đoạn chỉ chạm nhau ở đầu mút.

**Ví dụ 2**

- Input: `intervals = [[1,2],[1,2],[1,2]]`
- Output: `2`
- Giải thích: ba đoạn giống hệt nhau, phải xoá hai đoạn để chỉ còn một.

**Ví dụ 3**

- Input: `intervals = [[1,2],[2,3]]`
- Output: `0`
- Giải thích: đã không chồng lấn sẵn, không cần xoá.

**Ràng buộc:**

- `1 <= intervals.length <= 10^5`
- `intervals[i].length == 2`
- `-5 * 10^4 <= start_i < end_i <= 5 * 10^4`

Với `n` tới `10^5`, cách vét cạn (`2^n`) hay DP `O(n²)` đều quá chậm; ta cần `O(n log n)`.

In [1]:
from typing import List
import bisect
import itertools
import random

examples = [
    ([[1, 2], [2, 3], [3, 4], [1, 3]], 1),
    ([[1, 2], [1, 2], [1, 2]], 2),
    ([[1, 2], [2, 3]], 0),
]
for intervals, expected in examples:
    print(intervals, "-> kỳ vọng", expected)

[[1, 2], [2, 3], [3, 4], [1, 3]] -> kỳ vọng 1
[[1, 2], [1, 2], [1, 2]] -> kỳ vọng 2
[[1, 2], [2, 3]] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Hai đoạn chồng lấn khi nào?

Hai đoạn `[a, b]` và `[c, d]` **chồng lấn** (có phần chung dài hơn một điểm) khi `a < d` **và** `c < b`. Nếu chỉ có `b == c` (hoặc `d == a`) thì chúng **chạm nhau** nhưng **không chồng**. Vì vậy một đoạn mới `[l, r]` đặt được sau đoạn trước kết thúc ở `pre` khi **`l >= pre`** (dấu `>=`, không phải `>`).

In [2]:
def overlap(a, b):
    """True nếu hai đoạn a=[l1,r1], b=[l2,r2] chồng lấn thật sự."""
    return a[0] < b[1] and b[0] < a[1]


print(overlap([1, 2], [2, 3]))     # False (chỉ chạm tại điểm 2)
print(overlap([1, 3], [2, 4]))     # True
print(overlap([1, 10], [3, 4]))    # True (đoạn nhỏ nằm trọn trong đoạn lớn)
print(overlap([1, 2], [3, 4]))     # False

False
True
True
False


### 3.2. Sắp xếp với `key` và `lambda`

`intervals.sort(key=lambda x: x[1])` sắp **tại chỗ** các đoạn theo **phần tử thứ hai** (điểm kết thúc). `lambda x: x[1]` là hàm nhỏ không tên: nhận `x` (một đoạn) và trả về `x[1]`. `sorted(intervals, key=...)` thì **trả về danh sách mới**, không đụng đến bản gốc. Python dùng sắp xếp **ổn định**: hai đoạn cùng khoá giữ nguyên thứ tự ban đầu.

In [3]:
a = [[1, 2], [2, 3], [3, 4], [1, 3]]
print(sorted(a, key=lambda x: x[1]))     # theo điểm kết thúc
print(sorted(a, key=lambda x: x[0]))     # theo điểm bắt đầu
print(a)                                 # sorted không làm đổi a
a.sort(key=lambda x: x[1])
print(a)                                 # sort đã đổi chính a

[[1, 2], [2, 3], [1, 3], [3, 4]]
[[1, 2], [1, 3], [2, 3], [3, 4]]
[[1, 2], [2, 3], [3, 4], [1, 3]]
[[1, 2], [2, 3], [1, 3], [3, 4]]


### 3.3. Giải nén cặp trong vòng lặp

Mỗi đoạn là một list hai phần tử, nên có thể viết `for l, r in intervals:` để lấy ngay hai đầu mút thay vì `x[0]`, `x[1]`. Cách này đọc dễ hơn.

In [4]:
for l, r in [[1, 2], [2, 3], [3, 4]]:
    print("bắt đầu", l, "kết thúc", r, "độ dài", r - l)

bắt đầu 1 kết thúc 2 độ dài 1
bắt đầu 2 kết thúc 3 độ dài 1
bắt đầu 3 kết thúc 4 độ dài 1


### 3.4. `float('-inf')`: giá trị "nhỏ hơn mọi số"

Trước khi giữ đoạn nào, chưa có "điểm kết thúc của đoạn trước". Ta đặt `pre = float('-inf')` để **đoạn đầu tiên luôn thoả** `l >= pre`. (Trên LeetCode có thể dùng `-inf` nếu đã `from math import inf`; trong notebook ta viết `float('-inf')` cho rõ.)

In [5]:
pre = float("-inf")
print(pre < -10**9, pre <= 5, max(pre, 3))     # True True 3

True True 3


### 3.5. Bitmask: duyệt mọi tập con của `n` phần tử

Với `n` phần tử, mỗi số `mask` từ `0` đến `2^n - 1` mã hoá một tập con: bit thứ `i` bằng `1` nghĩa là **chọn** phần tử `i`. Kiểm tra bằng `mask >> i & 1`. Có `2^n` tập con nên chỉ dùng được khi `n` nhỏ (cỡ `<= 15`) — đủ để làm **oracle** kiểm chứng.

In [6]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    chosen = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(f"mask={mask:03b} -> {chosen}")

mask=000 -> []
mask=001 -> ['a']
mask=010 -> ['b']
mask=011 -> ['a', 'b']
mask=100 -> ['c']
mask=101 -> ['a', 'c']
mask=110 -> ['b', 'c']
mask=111 -> ['a', 'b', 'c']


### 3.6. `bisect`: tìm nhị phân trên danh sách đã sắp

`bisect.bisect_right(arr, x)` trả về vị trí chèn `x` vào sau mọi phần tử `<= x`; nghĩa là có đúng bấy nhiêu phần tử `<= x`. `bisect_left` thì vị trí đứng trước các phần tử bằng `x`. Mỗi lần gọi `O(log n)`.

In [7]:
arr = [1, 3, 3, 5, 8]
print(bisect.bisect_left(arr, 3))     # 1 (trước các số 3)
print(bisect.bisect_right(arr, 3))    # 3 (sau các số 3)
print(bisect.bisect_right(arr, 0))    # 0
print(bisect.bisect_right(arr, 9))    # 5

1
3
0
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy nghĩ như **xếp lịch họp trong một phòng**: mỗi đoạn là một cuộc họp, hai cuộc họp trùng giờ thì không xếp chung được. "Xoá ít nhất" = "**giữ được nhiều cuộc họp nhất**" (rồi lấy tổng trừ đi số giữ).

Với `[[1,2],[2,3],[3,4],[1,3]]` (n = 4):

| Tập giữ | Có rời nhau? | Số giữ |
|---|---|---|
| `[1,2],[2,3],[3,4]` | có | 3 |
| `[1,3],[3,4]` | có | 2 |
| `[1,2],[1,3]` | không | - |

Giữ tối đa `3`, vậy phải xoá `4 - 3 = 1`. Công thức: **đáp án = n − (số đoạn rời nhau lớn nhất)**.

In [8]:
intervals = [[1, 2], [2, 3], [3, 4], [1, 3]]
n = len(intervals)
best = 3                                   # đã tính tay: giữ [1,2],[2,3],[3,4]
print("phải xoá:", n - best)               # 1

phải xoá: 1


## 5. Cách 1 — Brute Force / baseline: thử mọi tập con

**Ý tưởng.** Với mỗi tập con (bitmask), sắp các đoạn được chọn theo `start`, kiểm tra từng cặp **liền kề** có chạm/rời nhau không (`prev_end <= next_start`). Nếu hợp lệ, cập nhật số đoạn giữ lớn nhất. Đáp án `n - best`.

In [9]:
def erase_bruteforce(intervals):
    n = len(intervals)
    best = 0
    for mask in range(1 << n):
        chosen = sorted(intervals[i] for i in range(n) if mask >> i & 1)
        ok = True
        for k in range(1, len(chosen)):
            if chosen[k][0] < chosen[k - 1][1]:      # đoạn sau bắt đầu trước khi đoạn trước kết thúc
                ok = False
                break
        if ok:
            best = max(best, len(chosen))
    return n - best


for iv, expected in examples:
    print(erase_bruteforce(iv), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Có `2^n` tập con, mỗi tập sắp xếp và kiểm tra `O(n log n)` ⇒ **Time `O(2^n · n log n)`**, **Space `O(n)`**. Chỉ chạy được với `n <= ~15`.

**Nút thắt:** ta thử cả những tập con vô vọng. Nếu sắp các đoạn theo một thứ tự cố định thì có thể xây tập con **dần dần** và chỉ nhớ "đoạn cuối đã chọn" thay vì cả tập.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp theo điểm kết thúc, rồi tham lam.** Sắp các đoạn theo `end` tăng dần. Đoạn có `end` nhỏ nhất **luôn có một lời giải tối ưu chứa nó**: giả sử lời giải tối ưu `S` không chứa nó, ta thay đoạn đầu tiên của `S` bằng đoạn kết thúc sớm nhất này — vì nó kết thúc **không muộn hơn**, nên không đụng vào đoạn nào khác của `S`, số đoạn giữ không đổi (**exchange argument**). Sau khi giữ nó, bỏ mọi đoạn chồng với nó, lặp lại với phần còn lại.

Khi cài đặt: duyệt theo `end` tăng dần, giữ biến `pre` = điểm kết thúc của đoạn cuối đã giữ. Đoạn `[l, r]` được giữ nếu `l >= pre`; khi đó `pre = r`. Ngược lại đoạn đó bị xoá (đếm `+1`).

In [10]:
intervals = [[1, 2], [2, 3], [3, 4], [1, 3]]
intervals_sorted = sorted(intervals, key=lambda x: x[1])
print("sau khi sắp theo end:", intervals_sorted)
pre = float("-inf")
removed = 0
for l, r in intervals_sorted:
    if l >= pre:
        print(f"  giữ [{l},{r}]  (l={l} >= pre={pre})")
        pre = r
    else:
        print(f"  xoá [{l},{r}]  (l={l} < pre={pre})")
        removed += 1
print("số đoạn xoá:", removed)

sau khi sắp theo end: [[1, 2], [2, 3], [1, 3], [3, 4]]
  giữ [1,2]  (l=1 >= pre=-inf)
  giữ [2,3]  (l=2 >= pre=2)
  xoá [1,3]  (l=1 < pre=3)
  giữ [3,4]  (l=3 >= pre=3)
số đoạn xoá: 1


## 7. Các cách tiếp theo

### Cách 2 — Quy hoạch động kiểu LIS (`O(n²)`)

**Giải thích từng đoạn.**

1. Sắp đoạn theo `start` tăng dần.
2. `dp[i]` = số đoạn rời nhau **nhiều nhất** trong các đoạn `0..i` mà **đoạn `i` được giữ làm đoạn cuối**.
3. `dp[i] = 1 + max(dp[j])` với mọi `j < i` mà `intervals[j][1] <= intervals[i][0]` (đoạn `j` kết thúc trước hoặc đúng lúc đoạn `i` bắt đầu); nếu không có `j` nào thì `dp[i] = 1`.
4. Số giữ tối đa là `max(dp)`; đáp án `n - max(dp)`.

Đây chính là bài **Longest Increasing Subsequence** biến dạng: "tăng" nghĩa là `end` của đoạn trước `<=` `start` của đoạn sau.

In [11]:
def erase_dp_n2(intervals):
    arr = sorted(intervals)                    # sắp theo start (rồi theo end)
    n = len(arr)
    dp = [1] * n
    for i in range(n):
        for j in range(i):
            if arr[j][1] <= arr[i][0]:         # đoạn j kết thúc trước khi i bắt đầu
                dp[i] = max(dp[i], dp[j] + 1)
    return n - max(dp)


for iv, expected in examples:
    print(erase_dp_n2(iv), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp). Lưu ý: O(n^2) sẽ TLE với n = 10^5.
class SolutionDP:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        arr = sorted(intervals)
        n = len(arr)
        dp = [1] * n
        for i in range(n):
            for j in range(i):
                if arr[j][1] <= arr[i][0]:
                    dp[i] = max(dp[i], dp[j] + 1)
        return n - max(dp)


print(SolutionDP().eraseOverlapIntervals([[1, 2], [2, 3], [3, 4], [1, 3]]))     # 1

1


**Độ phức tạp Cách 2:** **Time `O(n²)`** (hai vòng lồng), **Space `O(n)`** cho `dp`. Đúng nhưng chậm với `n = 10^5`.

### Cách 3 — Tham lam: sắp theo điểm kết thúc, giữ đoạn kết thúc sớm nhất (`O(n log n)`)

**Giải thích từng đoạn.**

1. `intervals.sort(key=lambda x: x[1])`: sắp theo `end` tăng dần.
2. `pre = -inf`: điểm kết thúc của đoạn đã giữ gần nhất; `keep = 0`.
3. Với mỗi `(l, r)`: nếu `l >= pre` thì **giữ** (`keep += 1`, `pre = r`); ngược lại bỏ qua.
4. Trả về `n - keep`.

In [13]:
def erase_greedy_end(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    keep = 0
    for l, r in arr:
        if l >= pre:
            keep += 1
            pre = r
    return len(arr) - keep


for iv, expected in examples:
    print(erase_greedy_end(iv), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [14]:
# Code nộp LeetCode cho Cách 3
class SolutionGreedyEnd:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: x[1])
        ans = len(intervals)
        pre = float("-inf")
        for l, r in intervals:
            if pre <= l:
                ans -= 1
                pre = r
        return ans


print(SolutionGreedyEnd().eraseOverlapIntervals([[1, 2], [1, 2], [1, 2]]))      # 2

2


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (sắp xếp chiếm phần lớn; duyệt chỉ `O(n)`), **Space `O(log n)`** hoặc `O(n)` tuỳ cách Python sắp xếp.

### Cách 4 — Tham lam: sắp theo điểm bắt đầu, gặp chồng lấn thì xoá đoạn kết thúc muộn hơn

**Giải thích từng đoạn.**

1. Sắp theo `start` tăng dần.
2. Giữ `pre_end` = điểm kết thúc của đoạn đang giữ gần nhất. Với đoạn mới `(l, r)`:
   - nếu `l >= pre_end`: không chồng, giữ nó, `pre_end = r`.
   - nếu `l < pre_end`: **chồng** ⇒ phải xoá **một trong hai** (`removed += 1`). Ta xoá đoạn có `end` **lớn hơn** (vì đoạn kết thúc sớm hơn để lại nhiều chỗ hơn), tức `pre_end = min(pre_end, r)`.
3. Trả về `removed`.

Điểm khác Cách 3: ở đây ta **thay đổi quyết định** (giữ đoạn có `end` nhỏ hơn trong hai đoạn đang chồng) thay vì dựa vào thứ tự sắp xếp.

In [15]:
def erase_greedy_start(intervals):
    arr = sorted(intervals)                    # theo start
    pre_end = arr[0][1]
    removed = 0
    for l, r in arr[1:]:
        if l >= pre_end:
            pre_end = r                        # giữ đoạn mới
        else:
            removed += 1                       # chồng lấn: bỏ một đoạn
            pre_end = min(pre_end, r)          # giữ đoạn kết thúc sớm hơn
    return removed


for iv, expected in examples:
    print(erase_greedy_start(iv), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionGreedyStart:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort()
        pre_end = intervals[0][1]
        removed = 0
        for l, r in intervals[1:]:
            if l >= pre_end:
                pre_end = r
            else:
                removed += 1
                pre_end = min(pre_end, r)
        return removed


print(SolutionGreedyStart().eraseOverlapIntervals([[1, 2], [2, 3], [3, 4], [1, 3]]))     # 1

1


**Độ phức tạp Cách 4:** **Time `O(n log n)`**, **Space `O(n)`** do `arr[1:]` tạo bản sao (có thể dùng chỉ số để còn `O(1)` thêm).

### Cách 5 — DP + tìm kiếm nhị phân (`O(n log n)`)

**Giải thích từng đoạn.**

1. Sắp theo `end` tăng dần; tách mảng `ends` và `starts` tương ứng.
2. `dp[i]` = số đoạn rời nhau lớn nhất **chỉ dùng `i` đoạn đầu** (theo thứ tự `end`).
3. Với đoạn thứ `i` (0-based): hoặc **bỏ** nó ⇒ `dp[i]`; hoặc **giữ** nó ⇒ `1 + dp[k]` với `k` = số đoạn trong `i` đoạn đầu có `end <= start_i`. Vì `ends` đã sắp tăng, `k = bisect_right(ends, start_i, 0, i)`.
4. `dp[i+1] = max(dp[i], 1 + dp[k])`. Giữ tối đa `dp[n]`.

Đây là dạng **weighted interval scheduling** (trọng số mỗi đoạn là `1`), tổng quát hơn Cách 3 vì còn giải được khi mỗi đoạn có giá trị khác nhau.

In [17]:
def erase_dp_bisect(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    n = len(arr)
    ends = [r for _, r in arr]
    dp = [0] * (n + 1)
    for i in range(n):
        l, r = arr[i]
        k = bisect.bisect_right(ends, l, 0, i)     # số đoạn trong 0..i-1 có end <= l
        dp[i + 1] = max(dp[i], 1 + dp[k])
    return n - dp[n]


for iv, expected in examples:
    print(erase_dp_bisect(iv), "(kỳ vọng", expected, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


In [18]:
# Code nộp LeetCode cho Cách 5
class SolutionDPBisect:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: x[1])
        n = len(intervals)
        ends = [r for _, r in intervals]
        dp = [0] * (n + 1)
        for i, (l, _) in enumerate(intervals):
            k = bisect.bisect_right(ends, l, 0, i)
            dp[i + 1] = max(dp[i], 1 + dp[k])
        return n - dp[n]


print(SolutionDPBisect().eraseOverlapIntervals([[1, 2], [2, 3]]))     # 0

0


**Độ phức tạp Cách 5:** **Time `O(n log n)`** (sắp xếp + `n` lần `bisect`), **Space `O(n)`** cho `dp` và `ends`.

## 8. TRACE từng bước

Ta trace Cách 3 (tham lam theo `end`) trên `[[1,2],[2,3],[3,4],[1,3]]` và một ví dụ khó hơn có đoạn dài bao trùm: `[[1,100],[11,22],[1,11],[2,12]]`.

In [19]:
def trace_greedy_end(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    print("đã sắp theo end:", arr)
    pre = float("-inf")
    keep = 0
    for step, (l, r) in enumerate(arr, 1):
        if l >= pre:
            action = "GIỮ  "
            keep += 1
            pre = r
        else:
            action = "XOÁ  "
        print(f"bước {step}: đoạn [{l},{r}] -> {action} | pre={pre} | giữ={keep}, xoá={step - keep}")
    return len(arr) - keep


assert trace_greedy_end([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
print()
assert trace_greedy_end([[1, 100], [11, 22], [1, 11], [2, 12]]) == 2

đã sắp theo end: [[1, 2], [2, 3], [1, 3], [3, 4]]
bước 1: đoạn [1,2] -> GIỮ   | pre=2 | giữ=1, xoá=0
bước 2: đoạn [2,3] -> GIỮ   | pre=3 | giữ=2, xoá=0
bước 3: đoạn [1,3] -> XOÁ   | pre=3 | giữ=2, xoá=1
bước 4: đoạn [3,4] -> GIỮ   | pre=4 | giữ=3, xoá=1

đã sắp theo end: [[1, 11], [2, 12], [11, 22], [1, 100]]
bước 1: đoạn [1,11] -> GIỮ   | pre=11 | giữ=1, xoá=0
bước 2: đoạn [2,12] -> XOÁ   | pre=11 | giữ=1, xoá=1
bước 3: đoạn [11,22] -> GIỮ   | pre=22 | giữ=2, xoá=1
bước 4: đoạn [1,100] -> XOÁ   | pre=22 | giữ=2, xoá=2


In [20]:
def trace_greedy_start(intervals):
    arr = sorted(intervals)
    print("đã sắp theo start:", arr)
    pre_end = arr[0][1]
    removed = 0
    print(f"khởi tạo: giữ {arr[0]}, pre_end={pre_end}")
    for l, r in arr[1:]:
        if l >= pre_end:
            print(f"  [{l},{r}]: l={l} >= pre_end={pre_end} -> giữ, pre_end={r}")
            pre_end = r
        else:
            removed += 1
            new_end = min(pre_end, r)
            print(f"  [{l},{r}]: l={l} < pre_end={pre_end} -> chồng, xoá 1 (xoá {max(pre_end, r)}), pre_end={new_end}")
            pre_end = new_end
    return removed


assert trace_greedy_start([[1, 100], [11, 22], [1, 11], [2, 12]]) == 2

đã sắp theo start: [[1, 11], [1, 100], [2, 12], [11, 22]]
khởi tạo: giữ [1, 11], pre_end=11
  [1,100]: l=1 < pre_end=11 -> chồng, xoá 1 (xoá 100), pre_end=11
  [2,12]: l=2 < pre_end=11 -> chồng, xoá 1 (xoá 12), pre_end=11
  [11,22]: l=11 >= pre_end=11 -> giữ, pre_end=22


Bảng trace Cách 3 với `[[1,2],[2,3],[3,4],[1,3]]` (đã sắp theo `end`: `[1,2],[2,3],[1,3],[3,4]`):

| Bước | Đoạn `[l,r]` | `pre` trước | `l >= pre`? | Hành động | `pre` sau | Số xoá |
|---|---|---|---|---|---|---|
| 1 | `[1,2]` | `-inf` | có | giữ | 2 | 0 |
| 2 | `[2,3]` | 2 | có (`2 >= 2`) | giữ | 3 | 0 |
| 3 | `[1,3]` | 3 | không (`1 < 3`) | xoá | 3 | 1 |
| 4 | `[3,4]` | 3 | có (`3 >= 3`) | giữ | 4 | 1 |

Đáp án `1`. Chú ý bước 2 và 4: `l == pre` vẫn **giữ** vì chạm đầu mút không phải chồng lấn.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (sắp theo `end` + tham lam): ngắn, `O(n log n)`.

In [21]:
class Solution:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: x[1])      # sắp theo điểm kết thúc
        ans = len(intervals)                    # ban đầu coi như phải xoá hết, mỗi lần giữ được thì trừ 1
        pre = float("-inf")                     # điểm kết thúc của đoạn đã giữ gần nhất
        for l, r in intervals:
            if pre <= l:                        # không chồng lấn (chạm đầu mút vẫn được)
                ans -= 1
                pre = r
        return ans


sol = Solution()
for iv, expected in examples:
    print(iv, "->", sol.eraseOverlapIntervals([x[:] for x in iv]))

[[1, 2], [2, 3], [3, 4], [1, 3]] -> 1
[[1, 2], [1, 2], [1, 2]] -> 2
[[1, 2], [2, 3]] -> 0


## 10. Vì sao thuật toán đúng

- **Đổi bài toán:** xoá ít nhất ⇔ giữ nhiều nhất các đoạn đôi một không chồng lấn. Đáp án = `n − giữ tối đa`.
- **Lựa chọn tham lam an toàn:** gọi `A` là đoạn có `end` nhỏ nhất. Với mọi tập hợp tối ưu `S` bất kỳ, ta có thể thay đoạn đầu tiên của `S` (theo thứ tự thời gian) bằng `A`: `A` kết thúc không muộn hơn đoạn đó nên không chồng với bất cứ đoạn nào còn lại trong `S`. Số đoạn giữ không giảm, nên tồn tại lời giải tối ưu chứa `A`.
- **Quy nạp:** sau khi chốt `A`, các đoạn chồng với `A` bị loại; phần còn lại (những đoạn có `start >= end_A`) là bài toán cùng dạng nhỏ hơn. Duyệt theo `end` tăng dần với `pre` chính là làm việc này lần lượt.
- **Điều kiện `>=`:** đề coi đoạn chạm đầu mút là hợp lệ nên so sánh `l >= pre`.
- **Đối chiếu độc lập:** trong mục 12 ta so Cách 3 với vét cạn tập con và với DP trên hàng nghìn input ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(intervals)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn tập con | `O(2^n · n log n)` | `O(n)` |
| 2. DP kiểu LIS | `O(n²)` | `O(n)` |
| 3. Tham lam theo `end` | `O(n log n)` | `O(log n)` |
| 4. Tham lam theo `start` | `O(n log n)` | `O(n)` (hoặc `O(1)` thêm nếu dùng chỉ số) |
| 5. DP + `bisect` | `O(n log n)` | `O(n)` |

Với `n = 10^5`: `n log n ≈ 1.7 · 10^6` thao tác (nhanh), còn `n² = 10^10` (quá chậm).

## 12. Bộ test

Ta gom các cách vào `impls`. Vì hàm của LeetCode sắp xếp **tại chỗ**, mỗi lần gọi ta truyền **bản sao**.

In [22]:
sol = Solution()
impls = {
    "dp_n2": erase_dp_n2,
    "greedy_end": erase_greedy_end,
    "greedy_start": erase_greedy_start,
    "dp_bisect": erase_dp_bisect,
    "leetcode": lambda iv: Solution().eraseOverlapIntervals([x[:] for x in iv]),
    "submit_dp": lambda iv: SolutionDP().eraseOverlapIntervals([x[:] for x in iv]),
    "submit_start": lambda iv: SolutionGreedyStart().eraseOverlapIntervals([x[:] for x in iv]),
    "submit_bisect": lambda iv: SolutionDPBisect().eraseOverlapIntervals([x[:] for x in iv]),
}


def check_all(intervals, expected, label, brute=True):
    if brute:
        assert erase_bruteforce(intervals) == expected, (label, "bruteforce")
    for key, fn in impls.items():
        actual = fn(intervals)
        assert actual == expected, (label, key, intervals, actual, expected)
    print("PASS", label)


for idx, (iv, expected) in enumerate(examples, 1):
    check_all(iv, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [23]:
check_all([[0, 1]], 0, "một đoạn duy nhất")
check_all([[1, 2], [3, 4]], 0, "hai đoạn rời nhau")
check_all([[1, 2], [2, 3]], 0, "hai đoạn chạm đầu mút")
check_all([[1, 3], [2, 4]], 1, "hai đoạn chồng nhau")
check_all([[1, 10], [2, 3]], 1, "một đoạn chứa đoạn kia")
check_all([[1, 100], [11, 22], [1, 11], [2, 12]], 2, "đoạn dài bao trùm")
check_all([[-5, -3], [-4, -2], [-3, 0], [0, 5]], 1, "có toạ độ âm")
check_all([[1, 5]] * 6, 5, "nhiều đoạn giống hệt", brute=True)
check_all([[0, 2], [1, 3], [2, 4], [3, 5], [4, 6]], 2, "dãy bậc thang chồng nhau")
check_all([[-50000, 50000], [-50000, 0], [0, 50000]], 1, "đúng biên toạ độ")

PASS một đoạn duy nhất
PASS hai đoạn rời nhau
PASS hai đoạn chạm đầu mút
PASS hai đoạn chồng nhau
PASS một đoạn chứa đoạn kia
PASS đoạn dài bao trùm
PASS có toạ độ âm
PASS nhiều đoạn giống hệt
PASS dãy bậc thang chồng nhau
PASS đúng biên toạ độ


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi tham lam phổ biến: (a) **sắp theo `start` rồi luôn giữ đoạn đã giữ trước đó** (không đổi sang đoạn kết thúc sớm hơn); (b) **sắp theo độ dài** rồi giữ đoạn ngắn trước. Cả hai đều có phản ví dụ.

In [24]:
def wrong_sort_by_start_keep_first(intervals):
    arr = sorted(intervals)
    pre = float("-inf")
    keep = 0
    for l, r in arr:
        if l >= pre:
            keep += 1
            pre = r              # SAI: không xét đoạn sau có end nhỏ hơn
    return len(arr) - keep


def wrong_sort_by_length(intervals):
    arr = sorted(intervals, key=lambda x: x[1] - x[0])
    chosen = []
    for l, r in arr:
        if all(r <= a or l >= b for a, b in chosen):
            chosen.append((l, r))
    return len(arr) - len(chosen)


bad1 = [[1, 100], [2, 3], [4, 5]]
print("sort theo start, giữ đầu:", wrong_sort_by_start_keep_first(bad1), "| đúng:", erase_greedy_end(bad1))
assert wrong_sort_by_start_keep_first(bad1) != erase_greedy_end(bad1)

bad2 = [[1, 5], [4, 7], [6, 10]]      # đoạn giữa ngắn nhất nhưng chặn cả hai đoạn kia
print("sort theo độ dài:", wrong_sort_by_length(bad2), "| đúng:", erase_greedy_end(bad2))
assert wrong_sort_by_length(bad2) != erase_greedy_end(bad2)

sort theo start, giữ đầu: 2 | đúng: 1
sort theo độ dài: 2 | đúng: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) và vét cạn

Sinh `intervals` đúng ràng buộc (`start < end`, toạ độ trong `[-5*10^4, 5*10^4]`). Với `n <= 10` ta đối chiếu thêm với vét cạn; với `n` lớn hơn ta so các cách `O(n log n)` với DP `O(n²)`. Thêm thuộc tính: đáp án luôn nằm trong `[0, n - 1]`, và sau khi xoá theo cách tham lam, tập còn lại thực sự rời nhau.

In [25]:
rng = random.Random(435)


def gen(n, lo, hi):
    out = []
    for _ in range(n):
        a = rng.randint(lo, hi - 1)
        b = rng.randint(a + 1, hi)
        out.append([a, b])
    return out


for trial in range(1500):
    n = rng.randint(1, 10)
    iv = gen(n, -6, 6)
    expected = erase_bruteforce(iv)
    for key, fn in impls.items():
        assert fn(iv) == expected, (key, iv)
    assert 0 <= expected <= n - 1
print("PASS 1500 input nhỏ (có vét cạn)")

for trial in range(300):
    n = rng.randint(30, 120)
    iv = gen(n, -50000, 50000) if trial % 2 else gen(n, 0, 60)
    ref = erase_dp_n2(iv)
    for key, fn in impls.items():
        assert fn(iv) == ref, (key, n)
print("PASS 300 input vừa so với DP O(n^2)")


def kept_set_is_valid(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    kept = []
    for l, r in arr:
        if l >= pre:
            kept.append([l, r])
            pre = r
    kept.sort()
    return all(kept[i][1] <= kept[i + 1][0] for i in range(len(kept) - 1)), len(kept)


for trial in range(300):
    iv = gen(rng.randint(1, 60), -30, 30)
    ok, k = kept_set_is_valid(iv)
    assert ok and len(iv) - k == erase_greedy_end(iv)
print("PASS tập giữ lại luôn rời nhau")

big = gen(100000, -50000, 50000)
print("n = 10^5 ->", Solution().eraseOverlapIntervals(big))

PASS 1500 input nhỏ (có vét cạn)
PASS 300 input vừa so với DP O(n^2)
PASS tập giữ lại luôn rời nhau


n = 10^5 -> 99512


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp theo `start` rồi chỉ giữ đoạn đến trước

In [26]:
def sai_sap_theo_start(intervals):
    intervals = sorted(intervals)
    pre = float("-inf")
    ans = 0
    for l, r in intervals:
        if l >= pre:
            pre = r
        else:
            ans += 1             # SAI: đoạn đang giữ có thể kết thúc rất muộn, đáng lẽ nên giữ đoạn mới
    return ans


iv = [[1, 100], [2, 3], [4, 5]]
print("sai:", sai_sap_theo_start(iv), "| đúng:", Solution().eraseOverlapIntervals([x[:] for x in iv]))

sai: 2 | đúng: 1


**Vì sao sai:** đoạn `[1,100]` bắt đầu sớm nhưng kết thúc rất muộn nên chặn mọi đoạn sau. Giữ nó làm ta xoá `2` đoạn, trong khi chỉ cần xoá chính nó (đáp án `1`). **Cách sửa:** sắp theo **`end`** (hoặc sắp theo `start` nhưng khi chồng thì giữ đoạn có `end` nhỏ hơn, Cách 4).

### Lỗi 2: Dùng `>` thay `>=` (coi đoạn chạm đầu mút là chồng lấn)

In [27]:
def sai_dau_lon_hon(intervals):
    intervals = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    ans = 0
    for l, r in intervals:
        if l > pre:              # SAI: phải là l >= pre
            pre = r
        else:
            ans += 1
    return ans


print("sai:", sai_dau_lon_hon([[1, 2], [2, 3]]), "| đúng:", Solution().eraseOverlapIntervals([[1, 2], [2, 3]]))
print("sai:", sai_dau_lon_hon([[1, 2], [2, 3], [3, 4], [1, 3]]), "| đúng: 1")

sai: 1 | đúng: 0
sai: 2 | đúng: 1


**Vì sao sai:** đề nói rõ `[1,2]` và `[2,3]` là **không chồng**. Dùng `l > pre` làm `[2,3]` bị xoá oan. **Cách sửa:** `l >= pre`.

### Lỗi 3: Quên rằng `sort()` sửa luôn mảng gốc

In [28]:
def dung_nhung_pha_vo_input(intervals):
    intervals.sort(key=lambda x: x[1])         # làm đổi thứ tự của list người gọi truyền vào
    return len(intervals)


data = [[1, 2], [2, 3], [3, 4], [1, 3]]
dung_nhung_pha_vo_input(data)
print("data sau khi gọi hàm:", data)           # không còn thứ tự ban đầu!
data2 = [[1, 2], [2, 3], [3, 4], [1, 3]]
sorted(data2, key=lambda x: x[1])
print("data2 sau khi dùng sorted:", data2)     # giữ nguyên

data sau khi gọi hàm: [[1, 2], [2, 3], [1, 3], [3, 4]]
data2 sau khi dùng sorted: [[1, 2], [2, 3], [3, 4], [1, 3]]


**Vì sao cần để ý:** trên LeetCode sửa mảng đầu vào thường chấp nhận được, nhưng khi bạn tự test và dùng lại cùng một biến cho nhiều hàm, kết quả có thể "lạ" vì dữ liệu đã bị sắp lại. **Cách sửa:** dùng `sorted(...)` để tạo bản mới, hoặc truyền `[x[:] for x in data]` khi gọi.

### Lỗi 4: Khởi tạo `pre = 0` thay vì `-inf`

In [29]:
def sai_pre_bang_0(intervals):
    intervals = sorted(intervals, key=lambda x: x[1])
    pre = 0                      # SAI: các đoạn có toạ độ âm sẽ bị coi là chồng
    ans = 0
    for l, r in intervals:
        if l >= pre:
            pre = r
        else:
            ans += 1
    return ans


iv = [[-5, -3], [-4, -2], [-3, 0]]
print("sai:", sai_pre_bang_0(iv), "| đúng:", Solution().eraseOverlapIntervals([x[:] for x in iv]))

sai: 3 | đúng: 1


**Vì sao sai:** đề cho phép toạ độ **âm** (`-5*10^4 <= start`). Với `pre = 0` đoạn đầu `[-5,-3]` có `l = -5 < 0` bị xoá oan. **Cách sửa:** `pre = float('-inf')` (hoặc lấy `pre = intervals[0][1]` cho đoạn đầu rồi duyệt tiếp).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Chọn nhiều đoạn nhất không chồng nhau**" / "**xoá ít nhất để hết chồng**" ⇒ **sắp theo `end`** rồi tham lam (Activity Selection / Interval Scheduling).
- "**Số phòng họp tối thiểu**", "**số lượng đoạn chồng tại một điểm**" ⇒ khác: dùng sweep line / heap, không phải bài này.
- Câu thần chú: *đoạn kết thúc sớm nhất để lại nhiều chỗ nhất*. Nếu bài hỏi theo **trọng số** (mỗi đoạn có giá trị) thì tham lam hỏng, chuyển sang **DP + bisect** (Cách 5).
- Chú ý điều kiện **biên**: đề nói chạm đầu mút có chồng hay không ⇒ chọn `>=` hay `>`.
- Khi viết tham lam, hãy tự sinh test nhỏ so với **vét cạn** để chắc rằng tiêu chí sắp xếp đúng.

## 15. Biến thể của bài

1. **Merge Intervals (56):** gộp các đoạn chồng nhau, cũng sắp theo `start`.
2. **Minimum Number of Arrows to Burst Balloons (452):** gần như đồng nhất — đoạn chồng chung một mũi tên; sắp theo `end`.
3. **Meeting Rooms (252) / Meeting Rooms II (253):** kiểm tra có chồng không / số phòng tối thiểu.
4. **Maximum Number of Events That Can Be Attended (1353):** mỗi sự kiện có khoảng ngày, chọn mỗi ngày một sự kiện — dùng heap.
5. **Maximum Profit in Job Scheduling (1235):** mỗi đoạn có lợi nhuận — chính là DP + bisect (Cách 5) với trọng số.
6. **Trả về chính các đoạn bị xoá** (không chỉ số lượng).

In [30]:
def removed_intervals(intervals):
    """Trả về danh sách các đoạn bị xoá (theo thuật toán tham lam sắp theo end)."""
    arr = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    removed = []
    for l, r in arr:
        if l >= pre:
            pre = r
        else:
            removed.append([l, r])
    return removed


def max_profit_scheduling(intervals, profits):
    """Biến thể có trọng số: chọn các đoạn rời nhau có tổng lợi nhuận lớn nhất (DP + bisect)."""
    items = sorted(zip(intervals, profits), key=lambda x: x[0][1])
    ends = [iv[1] for iv, _ in items]
    dp = [0] * (len(items) + 1)
    for i, ((l, r), p) in enumerate(items):
        k = bisect.bisect_right(ends, l, 0, i)
        dp[i + 1] = max(dp[i], dp[k] + p)
    return dp[-1]


def count_arrows(points):
    """Bài 452: các đoạn chồng (kể cả chạm đầu mút) bị bắn chung một mũi tên."""
    arr = sorted(points, key=lambda x: x[1])
    arrows, pre = 0, float("-inf")
    for l, r in arr:
        if l > pre:
            arrows += 1
            pre = r
    return arrows


print(removed_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))                  # [[1, 3]]
print(max_profit_scheduling([[1, 3], [2, 4], [3, 5], [3, 6]], [50, 10, 40, 70]))   # 120
print(count_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]))                     # 2
assert removed_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == [[1, 3]]
assert max_profit_scheduling([[1, 3], [2, 4], [3, 5], [3, 6]], [50, 10, 40, 70]) == 120
assert count_arrows([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2

[[1, 3]]
120
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn tập con | thử `2^n` tập, kiểm tra hợp lệ | `O(2^n · n log n)` | `O(n)` | oracle kiểm tra, `n <= 15` |
| 2. DP kiểu LIS | `dp[i]` = chuỗi rời dài nhất kết thúc tại `i` | `O(n²)` | `O(n)` | hiểu cấu trúc bài, `n` nhỏ |
| 3. Tham lam theo `end` | giữ đoạn kết thúc sớm nhất | `O(n log n)` | `O(log n)` | **nên nộp** |
| 4. Tham lam theo `start` | chồng thì bỏ đoạn `end` muộn | `O(n log n)` | `O(n)` | khi đã sắp theo `start` sẵn |
| 5. DP + `bisect` | `dp[i+1] = max(dp[i], 1 + dp[k])` | `O(n log n)` | `O(n)` | khi mỗi đoạn có trọng số |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `overlap([1,5],[5,9])`, `sorted([[3,4],[1,9],[2,2]], key=lambda x: x[1])`, `bisect.bisect_right([1,2,2,4], 2)`, `Solution().eraseOverlapIntervals([[1,2],[1,3],[2,3]])`.

**Bài 2 (tính tay).** Với `[[1,3],[2,4],[3,6],[5,7],[6,8]]`: sắp theo `end`, rồi chạy tham lam, ghi rõ từng bước giữ/xoá và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `erase_todo(intervals)` theo Cách 3.

In [31]:
def erase_todo(intervals):
    # TODO 1: sắp intervals theo điểm kết thúc (dùng sorted + key=lambda)
    # TODO 2: pre = float("-inf"); keep = 0
    # TODO 3: duyệt (l, r): nếu l >= pre thì keep += 1 và pre = r
    # TODO 4: trả về len(intervals) - keep
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `erase_greedy_start2(intervals)` theo Cách 4 **mà không tạo bản sao `arr[1:]`**: duyệt bằng chỉ số `for i in range(1, n)`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `kept_intervals(intervals)` trả về **danh sách các đoạn được giữ** (đã sắp theo `start`). (b) Viết `can_attend_all(intervals)` (bài Meeting Rooms 252): trả về `True` nếu không có hai đoạn nào chồng nhau.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `False` (chạm đầu mút), `[[2,2],[3,4],[1,9]]`, `3`, `1`. Bài 3: `sorted(intervals, key=lambda x: x[1])`.
- **Mức 2 (vừa):** Bài 2: sắp theo `end`: `[1,3],[2,4],[3,6],[5,7],[6,8]`. Giữ `[1,3]` (pre=3); `[2,4]` xoá; `[3,6]` giữ (3>=3, pre=6); `[5,7]` xoá; `[6,8]` giữ (6>=6). Giữ 3, xoá 2. Bài 4: gán `pre_end = arr[0][1]` rồi `for i in range(1, len(arr)): l, r = arr[i]`.
- **Mức 3 (sâu):** Bài 5(a): trong vòng tham lam theo `end`, thay vì đếm xoá, `append([l, r])` vào danh sách giữ khi `l >= pre`; cuối cùng `sort()`. 5(b): sắp theo `start` rồi chỉ cần so mỗi đoạn với đoạn liền trước: `arr[i][0] < arr[i-1][1]` thì có chồng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert overlap([1, 5], [5, 9]) is False
assert sorted([[3, 4], [1, 9], [2, 2]], key=lambda x: x[1]) == [[2, 2], [3, 4], [1, 9]]
assert bisect.bisect_right([1, 2, 2, 4], 2) == 3
assert Solution().eraseOverlapIntervals([[1, 2], [1, 3], [2, 3]]) == 1

# Bài 2
iv2 = [[1, 3], [2, 4], [3, 6], [5, 7], [6, 8]]
assert Solution().eraseOverlapIntervals([x[:] for x in iv2]) == 2


# Bài 3
def erase_todo_ans(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    keep = 0
    for l, r in arr:
        if l >= pre:
            keep += 1
            pre = r
    return len(intervals) - keep


for iv, expected in examples:
    assert erase_todo_ans(iv) == expected


# Bài 4
def erase_greedy_start2(intervals):
    arr = sorted(intervals)
    pre_end = arr[0][1]
    removed = 0
    for i in range(1, len(arr)):
        l, r = arr[i]
        if l >= pre_end:
            pre_end = r
        else:
            removed += 1
            pre_end = min(pre_end, r)
    return removed


for iv, expected in examples:
    assert erase_greedy_start2(iv) == expected


# Bài 5
def kept_intervals(intervals):
    arr = sorted(intervals, key=lambda x: x[1])
    pre = float("-inf")
    kept = []
    for l, r in arr:
        if l >= pre:
            kept.append([l, r])
            pre = r
    kept.sort()
    return kept


def can_attend_all(intervals):
    arr = sorted(intervals)
    return all(arr[i][0] >= arr[i - 1][1] for i in range(1, len(arr)))


assert kept_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]) == [[1, 2], [2, 3], [3, 4]]
assert can_attend_all([[0, 30], [5, 10], [15, 20]]) is False
assert can_attend_all([[7, 10], [2, 4]]) is True
assert can_attend_all([[1, 2], [2, 3]]) is True
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Interval Scheduling: giữ nhiều đoạn rời nhau nhất)

```python
class Solution:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: x[1])      # sắp theo điểm kết thúc
        ans = len(intervals)
        pre = float("-inf")
        for l, r in intervals:
            if pre <= l:                        # không chồng lấn (>=, chạm đầu mút vẫn được)
                ans -= 1                        # giữ đoạn này
                pre = r
        return ans
```

**Checklist**

- [ ] Hiểu đáp án = `n − (số đoạn rời nhau lớn nhất)`?
- [ ] Sắp theo **`end`** (không phải `start`, không phải độ dài)?
- [ ] Điều kiện giữ đoạn là `l >= pre` (đoạn chạm đầu mút hợp lệ)?
- [ ] `pre` khởi tạo là `-inf` (đề có toạ độ âm)?
- [ ] Đã thử `[[1,2],[2,3]]` (`0`), `[[1,2],[1,2],[1,2]]` (`2`), `[[1,100],[2,3],[4,5]]` (`1`) và một đoạn duy nhất (`0`)?
- [ ] Skeleton đúng `class Solution` / `def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int`?